In [1]:
from src.existingSimulator.simulator import simulateExperiment,exponential_lambda_search,calc_lambda

alpha,beta,beta_hat,time_spent = simulateExperiment(calc_lambda)

print("Single execution with binary+exponential lambda search:", alpha.item(), beta.item(), beta_hat.item(), time_spent)

Using binary search for lambda.
Single execution with binary+exponential lambda search: 0.4000083804130554 0.08481208980083466 0.08507156372070312 [0.1212012767791748]


In [2]:
import torch
import math
# NewtonImplentation
def calc_derivative(rho, lambdaVal, beta):
    exp_term = torch.exp(-lambdaVal * rho)
    sum_exp = torch.sum(exp_term, dim=1, keepdim=True)

    fraction_term = torch.sum(rho * exp_term, dim=1, keepdim=True) / (sum_exp + 1e-10)
    complete_middle_block = -rho - fraction_term
    fp = -torch.sum(
        beta * complete_middle_block * (torch.log2(beta + 1e-10) + 1/math.log(2)),
        dim=(1, 2, 3)
    )
    return fp

def calc_newton_lambda(rho, m, objective, xtol=1e-3, max_iter=20):
    print("Calculating lambda using Newton's method...")
    device = rho.device
    B = rho.shape[0]
    if isinstance(m, (int, float)):
        m = torch.full((B,), m, dtype=rho.dtype, device=device)
    lbdr, lbdh = exponential_lambda_search(rho, m, objective)
    lbd = (lbdr + lbdh) / 2.0
    
    for i in range(max_iter):
        beta, h_hat = objective(rho, lbd.view(-1, 1, 1, 1))
        f = h_hat - m
        f_derivated = calc_derivative(rho, lbd.view(-1, 1, 1, 1), beta)

        lbd = lbd - f / f_derivated
        lbd = torch.clamp(lbd, 1e-10, 1e10)
    _, h_hat = objective(rho, lbd.view(-1, 1, 1, 1))
    return lbd, h_hat
simulateExperiment(calc_newton_lambda)

Calculating lambda using Newton's method...


(tensor([0.3999]), tensor([0.0848]), tensor([0.0850]), [0.14538908004760742])